# `10 — Knuth–Morris–Pratt (KMP) algorithm`

We study:
- **String-searching problem**
- Naive solution (**O(NK)**)
- **Prefix-function** π (DP approach)
- **KMP** pattern matching using π
- Complexity: **O(N + K)**

Goal:
- Understand *why* we never re-check characters unnecessarily
- Be able to compute π by hand for a small pattern
- Visualize matching (shifts) step-by-step


## `1. String-searching problem`

Given:
- text **s** of length **N**
- pattern **p** of length **K**

Find all indices i such that: $s[i : i+K] == p$

Naive idea:
- try every shift i
- compare $p$ with $s[i...i+K-1]$

Worst-case: **$O(NK)$**.

In [1]:
# Naive search:
from typing import List


def naive_find_all(s: str, *, p: str, verbose: bool = True) -> List[int]:
    n: int = len(s)
    k: int = len(p)
    hits: List[int] = []

    for i in range(n - k + 1):
        if verbose:
            print("-" * 60)
            print(f"shift i={i}")
            print("s:", s)
            print("p:", " " * i + p)

        ok: bool = True
        for j in range(k):
            if s[i + j] != p[j]:
                ok = False
                if verbose:
                    print(f"mismatch at j={j}: s[{i+j}]={s[i+j]!r} != p[{j}]={p[j]!r}")
                break

        if ok:
            hits.append(i)
            if verbose:
                print("MATCH ✅")

    return hits


print(naive_find_all("ababacababa", p="aba", verbose=True))

------------------------------------------------------------
shift i=0
s: ababacababa
p: aba
MATCH ✅
------------------------------------------------------------
shift i=1
s: ababacababa
p:  aba
mismatch at j=0: s[1]='b' != p[0]='a'
------------------------------------------------------------
shift i=2
s: ababacababa
p:   aba
MATCH ✅
------------------------------------------------------------
shift i=3
s: ababacababa
p:    aba
mismatch at j=0: s[3]='b' != p[0]='a'
------------------------------------------------------------
shift i=4
s: ababacababa
p:     aba
mismatch at j=1: s[5]='c' != p[1]='b'
------------------------------------------------------------
shift i=5
s: ababacababa
p:      aba
mismatch at j=0: s[5]='c' != p[0]='a'
------------------------------------------------------------
shift i=6
s: ababacababa
p:       aba
MATCH ✅
------------------------------------------------------------
shift i=7
s: ababacababa
p:        aba
mismatch at j=0: s[7]='b' != p[0]='a'
--------------

## `2.1 Prefix-function π`

For a string t of length N:

**$π[i]$** = length of the longest proper prefix of $t[:i+1]$ that is also a suffix of $t[:i+1]$.

"Proper" means: not equal to the whole string.

Examples from lecture:
- π("abacaba") = 3
- π("aaaaaaa") = 6
- π("abcdefg") = 0
- π("abcabcabc") = 6

## `2.2 DP approach`

We compute π left-to-right.

Let:
- we are computing π[i]
- we start with candidate j = π[i-1]
- if t[i] matches t[j], we extend: π[i] = j+1
- otherwise, we "fallback" j = π[j-1] and try again
- if j becomes 0 and still mismatch, π[i]=0

Key point:
those fallbacks jump using previously computed π values.
That is why total time is linear.

In [2]:
from typing import List


def prefix_function_trace(t: str, *, verbose: bool = True) -> List[int]:
    """
    Returns pi array of length len(t).
    Verbose prints show how j falls back: j -> pi[j-1] -> ...
    """
    n: int = len(t)
    pi: List[int] = [0] * n

    if verbose:
        print("-" * 80)
        print("Prefix-function computation")
        print("t =", t)
        print("-" * 80)
        print("i | t[i] | start j | actions (fallbacks / match) | pi[i]")
        print("-" * 80)

    for i in range(1, n):
        j: int = pi[i - 1]
        actions: List[str] = [f"j=pi[{i-1}]={j}"]

        while j > 0 and t[i] != t[j]:
            actions.append(f"t[{i}]={t[i]} != t[{j}]={t[j]} -> j=pi[{j-1}]={pi[j-1]}")
            j = pi[j - 1]

        if t[i] == t[j]:
            j += 1
            actions.append(f"match t[{i}]={t[i]} == t[{j-1}]={t[j-1]} -> j={j}")
        else:
            actions.append(f"no match and j=0 -> j stays 0")

        pi[i] = j

        if verbose:
            print(f"{i:>1} |  {t[i]:>3} |    {pi[i-1]:>3}   | " + "; ".join(actions) + f" |  {pi[i]:>3}")

    if verbose:
        print("-" * 80)
        print("pi =", pi)
    return pi


prefix_function_trace("abacaba", verbose=True)


--------------------------------------------------------------------------------
Prefix-function computation
t = abacaba
--------------------------------------------------------------------------------
i | t[i] | start j | actions (fallbacks / match) | pi[i]
--------------------------------------------------------------------------------
1 |    b |      0   | j=pi[0]=0; no match and j=0 -> j stays 0 |    0
2 |    a |      0   | j=pi[1]=0; match t[2]=a == t[0]=a -> j=1 |    1
3 |    c |      1   | j=pi[2]=1; t[3]=c != t[1]=b -> j=pi[0]=0; no match and j=0 -> j stays 0 |    0
4 |    a |      0   | j=pi[3]=0; match t[4]=a == t[0]=a -> j=1 |    1
5 |    b |      1   | j=pi[4]=1; match t[5]=b == t[1]=b -> j=2 |    2
6 |    a |      2   | j=pi[5]=2; match t[6]=a == t[2]=a -> j=3 |    3
--------------------------------------------------------------------------------
pi = [0, 0, 1, 0, 1, 2, 3]


[0, 0, 1, 0, 1, 2, 3]

## `3.1 KMP idea`

Compute prefix-function on the concatenated string:

**t = p + '$' + s**

where '$' is a character not in the alphabet Σ.

If at some position i in t we have:
**π[i] == len(p)**

then we just matched the whole pattern ending at i.

From that, we can recover the start index in s.

In [3]:
from typing import List


def kmp_find_all_trace(s: str, *, p: str, verbose: bool = True) -> List[int]:
    """
    KMP via pi(p + '$' + s).
    Prints where matches are found and shows the pi values around them.
    """
    sep: str = "$"
    t: str = p + sep + s

    pi: List[int] = prefix_function_trace(t, verbose=False)

    k: int = len(p)
    hits: List[int] = []

    if verbose:
        print("-" * 80)
        print("KMP search using t = p + '$' + s")
        print("p =", p)
        print("s =", s)
        print("t =", t)
        print("-" * 80)
        print("i | t[i] | pi[i] | comment")
        print("-" * 80)

    for i in range(len(t)):
        comment: str = ""
        if pi[i] == k:
            # lecture formula: i - 2*len(p) - 1
            start_idx: int = i - 2 * k
            # Explanation: subtract pattern length (k) to get end in s-part,
            # plus subtract separator offset; in lecture it’s i - 2k - 1 due to pi array indexing style.
            # With our 0-based pi over t, the start is:
            start_idx = i - 2 * k
            hits.append(start_idx)
            comment = f"MATCH ends at t[{i}], start in s = {start_idx}"

        if verbose and i >= len(p) + 1:  # after separator, we are scanning s part
            print(f"{i:>2} |  {t[i]:>3} |  {pi[i]:>3} | {comment}")

    if verbose:
        print("-" * 80)
        print("Matches at indices:", hits)

    return hits


kmp_find_all_trace("ababacababa", p="aba", verbose=True)

--------------------------------------------------------------------------------
KMP search using t = p + '$' + s
p = aba
s = ababacababa
t = aba$ababacababa
--------------------------------------------------------------------------------
i | t[i] | pi[i] | comment
--------------------------------------------------------------------------------
 4 |    a |    1 | 
 5 |    b |    2 | 
 6 |    a |    3 | MATCH ends at t[6], start in s = 0
 7 |    b |    2 | 
 8 |    a |    3 | MATCH ends at t[8], start in s = 2
 9 |    c |    0 | 
10 |    a |    1 | 
11 |    b |    2 | 
12 |    a |    3 | MATCH ends at t[12], start in s = 6
13 |    b |    2 | 
14 |    a |    3 | MATCH ends at t[14], start in s = 8
--------------------------------------------------------------------------------
Matches at indices: [0, 2, 6, 8]


[0, 2, 6, 8]

In [4]:
from typing import List


def kmp_online_trace(s: str, *, p: str, verbose: bool = True) -> List[int]:
    """
    Standard KMP scan:
      - compute pi for pattern p
      - scan text s, maintaining j = current matched length
    """
    pi: List[int] = prefix_function_trace(p, verbose=False)
    hits: List[int] = []
    j: int = 0  # matched length in p

    if verbose:
        print("-" * 80)
        print("Online KMP scan")
        print("p =", p)
        print("pi(p) =", pi)
        print("s =", s)
        print("-" * 80)

    for i, ch in enumerate(s):
        if verbose:
            print(f"\ntext i={i}, s[i]={ch!r}, current matched j={j}")
            print("s:", s)
            print("p:", " " * (i - j) + p)
            if j > 0:
                print("   " + " " * i + "^ mismatch check / extend")

        while j > 0 and ch != p[j]:
            if verbose:
                print(f"  mismatch: s[i]={ch!r} != p[j]={p[j]!r} -> fallback j = pi[{j-1}]={pi[j-1]}")
            j = pi[j - 1]

        if ch == p[j]:
            j += 1
            if verbose:
                print(f"  match: extend j -> {j}")
        else:
            if verbose:
                print("  no match and j=0 -> stay j=0")

        if j == len(p):
            start: int = i - len(p) + 1
            hits.append(start)
            if verbose:
                print(f"  ✅ FULL MATCH ending at i={i}, start={start}")
            j = pi[j - 1]  # allow overlaps
            if verbose:
                print(f"  after match, set j = pi[last]={j} to allow overlaps")

    if verbose:
        print("\nMatches:", hits)

    return hits


kmp_online_trace("ababacababa", p="aba", verbose=True)

--------------------------------------------------------------------------------
Online KMP scan
p = aba
pi(p) = [0, 0, 1]
s = ababacababa
--------------------------------------------------------------------------------

text i=0, s[i]='a', current matched j=0
s: ababacababa
p: aba
  match: extend j -> 1

text i=1, s[i]='b', current matched j=1
s: ababacababa
p: aba
    ^ mismatch check / extend
  match: extend j -> 2

text i=2, s[i]='a', current matched j=2
s: ababacababa
p: aba
     ^ mismatch check / extend
  match: extend j -> 3
  ✅ FULL MATCH ending at i=2, start=0
  after match, set j = pi[last]=1 to allow overlaps

text i=3, s[i]='b', current matched j=1
s: ababacababa
p:   aba
      ^ mismatch check / extend
  match: extend j -> 2

text i=4, s[i]='a', current matched j=2
s: ababacababa
p:   aba
       ^ mismatch check / extend
  match: extend j -> 3
  ✅ FULL MATCH ending at i=4, start=2
  after match, set j = pi[last]=1 to allow overlaps

text i=5, s[i]='c', current matched j=1

[0, 2, 6, 8]

## **Complexity summary (oral exam)**

Let:
- N = len(s)
- K = len(p)

| Method | Time | Why |
|---|---:|---|
| Naive search | **O(NK)** | can restart comparisons many times |
| Compute π for a string of length L | **O(L)** | total fallback steps ≤ total forward steps |
| KMP search | **O(N + K)** | scan is linear + π is linear |


## **Real-world use cases**

1) **Searching a DNA motif** inside a genome string  
   - text = genome, pattern = motif

2) **Finding a keyword** in a log stream  
   - text = log line stream, pattern = "ERROR 503"

3) **Detecting repeated structure** in a string  
   - prefix-function reveals repetitions (periodicity), useful in compression / analysis